In [2]:
!pip install -r requirements.txt

In [3]:
import openai
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
MODEL = "gpt-4o-mini"
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

In [5]:
def get_today_weather(city: str) -> str:
    "지정된 도시의 현재 날씨, 온도 등의 데이터를 가져옵니다. 온도는 섭씨로 제공됩니다."

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units=metric&lang=kr"
    try:
        response = requests.get(url)
        response.raise_for_status() #
        data = response.json()

        if data.get("cod") == "404":
            return json.dumps({"error": f"도시 '{city}'의 날씨 데이터를 찾을 수 없습니다."})

        weather = data["weather"][0]["description"]
        temperature = data["main"]["temp"] 

        return json.dumps({
            "city": city,
            "weather": weather,
            "temperature_celsius": temperature
        })

    except requests.exceptions.HTTPError as e:
        return json.dumps({"error": f"API 요청 실패. 상태 코드: {response.status_code}, 상세: {e}"})
    except Exception as e:
        return json.dumps({"error": f"API 요청 중 예상치 못한 에러 발생: {e}"})

weather_tool_schema = {
    "type": "function", 
    "function" : 	{
        "name" : "get_today_weather", 
        "description" : "특정 도시의 날씨, 온도와 같은 기상 정보를 가져옵니다.",
        "parameters" : {
            "type" : "object",
            "properties" : {"city" : {"type" : "string", "description" : "기상 정보를 가져올 도시명(예: Seoul, Tokyo)"}},
            "required" : ["city"]
        }
    }
}

available_functions = {
    "get_today_weather" : get_today_weather
}

In [ ]:
messages = [{"role" : "user", "content": "오늘 서울 날씨 어때?"}]
tools = [weather_tool_schema]

print("1단계: LLM에 Tool 호출 요청")
response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
response_message = response.choices[0].message
messages.append(response_message)

tool_calls = response_message.tool_calls
if tool_calls:
    print(" LLM이 Tool 사용을 결정했습니다.")
    print("\n 2단계: 결정된 Tool을 실제로 실행")
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)
        print(f"- 실행 함수: {function_name}({', '.join([f'{k}={v}' for k, v in function_args.items()])})")

        function_response = function_to_call(**function_args)
        print(f"- 실행 결과: {function_response}")

        messages.append({"tool_call_id": tool_call.id, "role": "tool", "name": function_name, "content": function_response})

    print("\n 3단계: Tool 실행 결과를 바탕으로 최종 답변 생성")
    final_response = client.chat.completions.create(model=MODEL, messages=messages)
    print("\n===== Agent의 최종 답변 =====")
    agent_answer = final_response.choices[0].message.content
    print(agent_answer)
else:
    print(" LLM이 Tool 사용을 결정하지 않았습니다.")
    final_response = client.chat.completions.create(model=MODEL, messages=messages)
    print("\n===== Agent의 최종 답변 =====")
    print(final_response.choices[0].message.content)

1단계: LLM에 Tool 호출 요청
 LLM이 Tool 사용을 결정하지 않았습니다.

===== Agent의 최종 답변 =====
서강대학교(Sogang University)는 대한민국 서울특별시에 위치한 사립 대학으로, 1960년에 설립되었습니다. 예수회에서 운영하는 가톨릭 계열 대학으로, 인문학, 사회과학, 자연과학, 공학, 경영학 등 다양한 분야에서 학부 및 대학원 과정을 제공합니다.

서강대학교의 주요 특징은 다음과 같습니다:

1. **교육 이념**: "지혜와 사랑"이라는 교육 철학을 바탕으로, 학생들이 윤리적이고 사회적으로 책임감 있는 인재로 성장할 수 있도록 돕고 있습니다.

2. **우수한 교수진**: 국내외에서 인정받는 교수진이 학생들에게 연구와 교육의 질을 높이기 위해 힘쓰고 있습니다.

3. **캠퍼스**: 마포구에 위치한 캠퍼스는 아름다운 자연환경과 현대적인 시설로 유명하며, 학생들에게 쾌적한 학습 환경을 제공합니다.

4. **글로벌 경험**: 서강대학교는 해외 대학과의 교류 프로그램, 교환학생 프로그램 등을 통해 학생들에게 글로벌 경험을 쌓을 기회를 제공합니다.

5. **취업 지원**: 학생들의 진로와 취업을 지원하기 위한 다양한 프로그램과 네트워킹 기회를 제공하며, 많은 기업과의 산학 협력을 통해 실제적인 경험을 쌓는 기회를 제공합니다.

서강대학교는 한국 내에서 높은 명성을 가진 대학으로, 다양한 전공 분야에서 우수한 교육과 연구를 펼치고 있습니다.
